In [1]:
from albumentations.pytorch import ToTensorV2
from contextlib import nullcontext
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import PowerNorm, LinearSegmentedColormap
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from scipy.ndimage import sobel as scipy_sobel, uniform_filter
import torch, torchvision, timm
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2, os, glob, re, math, time, json as _json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#DEVICE = torch.device('cpu')  # GPU ist vom Training belegt
print(f"Device: {DEVICE}")
# =====================================================================
# V3.1 CONFIG
# =====================================================================
CONFIG = {
    'img_height': 480, 'img_width': 640,
    'num_det_classes': 40,
    'num_seg_classes': 6,
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 24,
    'tartan_fx': 320.0,
    'tartan_fy': 320.0,
    'tartan_baseline': 0.25,
    'seg_class_weights': [1.0, 3.0, 1.0, 2.0, 2.0, 0.5],
}

SEG_CLASS_NAMES = ['WALKABLE', 'STEP', 'WALL', 'OBSTACLE', 'VEGETATION', 'VOID']
SEG_COLORS = np.array([
    [255, 255, 255],    [255, 165, 0],  [100, 100, 200],
    [200, 50, 50],  [0, 150, 0],    [50, 50, 50]
], dtype=np.uint8)

ROBOT_CAT_IDS = [1,2,3,4,6,8,10,11,13,14,15,16,17,18,27,28,31,33,44,47,51,
                  62,63,64,65,67,70,72,73,75,76,77,78,79,81,82,84,85,86,88]
robot_cat_to_continuous = {cid: idx for idx, cid in enumerate(ROBOT_CAT_IDS)}

print(f"V3.1 Eval Config ready")

/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
V3.1 Eval Config ready


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math

# --- Hailo-8 Compatible Building Blocks (unchanged from V2.5) ---
class DWSepConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, bias=True, dilation=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride=stride, padding=padding,
                            dilation=dilation, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x):
        return self.pw(self.dw(x))

# --- FPN Neck (unchanged from V2.5) ---
FPN_CH = 64

class LightFPNNeck(nn.Module):
    def __init__(self, ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH):
        super().__init__()
        self.lat_s8  = nn.Conv2d(ch_s8,  fpn_ch, 1, bias=False)
        self.lat_s16 = nn.Conv2d(ch_s16, fpn_ch, 1, bias=False)
        self.lat_s32 = nn.Conv2d(ch_s32, fpn_ch, 1, bias=False)
        
        # 🚨 PATCH V3.0: RepConv statt DWSepConv für kräftigere Features!
        self.smooth_s8  = RepConv(fpn_ch, fpn_ch)
        self.smooth_s16 = RepConv(fpn_ch, fpn_ch)
        self.bu_s16 = RepConv(fpn_ch, fpn_ch, stride=2)
        self.bu_s32 = RepConv(fpn_ch, fpn_ch, stride=2)

    def forward(self, f_s8, f_s16, f_s32):
        p32 = self.lat_s32(f_s32)
        p16 = self.lat_s16(f_s16) + F.interpolate(p32, scale_factor=2, mode='nearest')
        p8  = self.lat_s8(f_s8) + F.interpolate(p16, scale_factor=2, mode='nearest')
        p8  = self.smooth_s8(p8)
        p16 = self.smooth_s16(p16) + self.bu_s16(p8)
        p32 = p32 + self.bu_s32(p16)
        return p8, p16, p32

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2  # Hidden Channels
        self.cv1 = nn.Sequential(nn.Conv2d(c1, c_, 1, 1, bias=False), nn.BatchNorm2d(c_), nn.SiLU(inplace=True))
        self.cv2 = nn.Sequential(nn.Conv2d(c_ * 4, c2, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.SiLU(inplace=True))
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        # Cat von Original + 3 MaxPool-Stufen
        return self.cv2(torch.cat((x, y1, y2, y3), 1))

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x_cat))

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

class GeometryStem(nn.Module):
    def __init__(self, in_ch=24, out_ch=32): # MobileNetV3 s4 hat oft 24 ch
        super().__init__()
        # Zwei Schichten für mehr Reife in den Features
        self.stem = nn.Sequential(
            RepConv(in_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=False),
            RepConv(out_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=False)
        )

    def forward(self, x):
        return self.stem(x)

import torch
import torch.nn as nn

class AddCoords(nn.Module):
    def forward(self, x):
        b, c, h, w = x.shape
        # Y- und X-Koordinaten-Grid erstellen (-1 bis 1)
        y_coords = torch.linspace(-1, 1, h, device=x.device).view(1, 1, h, 1).expand(b, 1, h, w)
        x_coords = torch.linspace(-1, 1, w, device=x.device).view(1, 1, 1, w).expand(b, 1, h, w)
        return torch.cat([x, y_coords, x_coords], dim=1)

class CoordConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, 
                 grid_h=None, grid_w=None):
        super().__init__()
        self.conv = nn.Conv2d(in_channels + 2, out_channels, kernel_size=kernel_size, 
                              stride=stride, padding=padding, bias=bias)
        
        if grid_h and grid_w:
            y = torch.linspace(-1, 1, grid_h).view(1, 1, grid_h, 1).expand(1, 1, grid_h, grid_w)
            x = torch.linspace(-1, 1, grid_w).view(1, 1, 1, grid_w).expand(1, 1, grid_h, grid_w)
            grid = torch.cat([y, x], dim=1)
            self.register_buffer('coord_grid', grid) # Wird Teil des Moduls, aber kein Gradient
        else:
            self.coord_grid = None

    def forward(self, x):
        if self.coord_grid is not None:
            # Für ONNX wird das hier zu einer einfachen Konstanten-Addition
            grid = self.coord_grid.expand(x.size(0), -1, -1, -1)
        else:
            # Fallback für Training/variable Auflösung
            b, c, h, w = x.shape
            y = torch.linspace(-1, 1, h, device=x.device).view(1, 1, h, 1).expand(b, 1, h, w)
            xc = torch.linspace(-1, 1, w, device=x.device).view(1, 1, 1, w).expand(b, 1, h, w)
            grid = torch.cat([y, xc], dim=1)
        return self.conv(torch.cat([x, grid], dim=1))
    
import torch
import torch.nn as nn
import torch.nn.functional as F

class RepConv(nn.Module):
    """
    Re-Parameterized Convolution:
    Training: 3x3 Conv + 1x1 Conv + Identity (Parallel)
    Inferenz: Eine einzige 3x3 Conv (zusammengefaltet)
    """
    def __init__(self, c1, c2, kernel_size=3, stride=1, padding=1, deploy=False):
        super().__init__()
        self.deploy = deploy
        self.c1 = c1
        self.c2 = c2
        self.stride = stride
        self.padding = padding
        self.act = nn.ReLU(inplace=True)

        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=True)
        else:
            # Identity Branch (nur möglich wenn Dimensionen gleich bleiben)
            self.rbr_identity = nn.BatchNorm2d(c1) if c2 == c1 and stride == 1 else None
            # 3x3 Branch
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False),
                nn.BatchNorm2d(c2)
            )
            # 1x1 Branch
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, stride, 0, bias=False),
                nn.BatchNorm2d(c2)
            )

    def forward(self, x):
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        
        id_out = 0 if self.rbr_identity is None else self.rbr_identity(x)
        return self.act(self.rbr_dense(x) + self.rbr_1x1(x) + id_out)

    def get_equivalent_kernel_bias(self):
        # 3x3 Kernel extrahieren
        kernel3x3, bias3x3 = self._fuse_bn_tensor(self.rbr_dense)
        # 1x1 Kernel extrahieren und auf 3x3 padden
        kernel1x1, bias1x1 = self._fuse_bn_tensor(self.rbr_1x1)
        kernel1x1 = F.pad(kernel1x1, [1, 1, 1, 1])
        # Identity Kernel erstellen (nur 1en in der Mitte)
        kernelid, biasid = self._fuse_bn_tensor(self.rbr_identity)
        
        return kernel3x3 + kernel1x1 + kernelid, bias3x3 + bias1x1 + biasid

    def _fuse_bn_tensor(self, branch):
        if branch is None:
            return torch.zeros((self.c2, self.c1, 3, 3), device=self.rbr_dense[0].weight.device), torch.zeros(self.c2, device=self.rbr_dense[0].weight.device)
        if isinstance(branch, nn.BatchNorm2d):
            # Trick: Fake-Kernel für Identity
            kernel = torch.zeros((self.c1, self.c1, 3, 3), device=branch.weight.device)
            for i in range(self.c1): kernel[i, i, 1, 1] = 1.0
            return self._fuse_bn(kernel, branch.running_mean, branch.running_var, branch.weight, branch.bias, branch.eps)
        else:
            return self._fuse_bn(branch[0].weight, branch[1].running_mean, branch[1].running_var, branch[1].weight, branch[1].bias, branch[1].eps)

    def _fuse_bn(self, kernel, mean, var, gamma, beta, eps):
        std = (var + eps).sqrt()
        t = (gamma / std).reshape(-1, 1, 1, 1)
        return kernel * t, beta - mean * gamma / std
        
    def switch_to_deploy(self):
        if self.deploy: return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(self.c1, self.c2, 3, self.stride, self.padding, bias=True)
        self.rbr_reparam.weight.data = kernel
        self.rbr_reparam.bias.data = bias
        # Lösche Trainings-Branches, um RAM zu befreien!
        for attr in ['rbr_dense', 'rbr_1x1', 'rbr_identity']:
            if hasattr(self, attr): delattr(self, attr)
        self.deploy = True
    
# --- Cost Volume (Korrigiert: Universelle Metrik) ---
class CoarseCostVolume(nn.Module):
    def __init__(self, max_disp, in_channels):
        super().__init__()
        self.max_disp = max_disp
        # ✅ FIX 1: Genau 1 Output-Kanal! Gleiche Metrik für jede Verschiebung.
        self.corr = nn.Conv2d(in_channels * 2, 1, 1, bias=True)
        
    def forward(self, feat_l, feat_r):
        B, C, H, W = feat_l.shape
        cost_slices = []
        for d in range(self.max_disp):
            if d == 0:
                cost_slices.append(torch.cat([feat_l, feat_r], dim=1))
            else:
                shifted = torch.zeros_like(feat_r)
                shifted[:, :, :, d:] = feat_r[:, :, :, :-d]
                cost_slices.append(torch.cat([feat_l, shifted], dim=1))
        
        cost = torch.stack(cost_slices, dim=2)
        B, C2, D, H, W = cost.shape
        
        # Umformen auf (B*D, C*2, H, W), damit die 1x1 Conv auf jeden Slice 
        # exakt gleich angewendet wird!
        cost = cost.permute(0, 2, 1, 3, 4).reshape(B * D, C2, H, W)
        
        out = self.corr(cost) # Shape: (B*D, 1, H, W)
        
        # Zurückformen auf das saubere Cost-Volume (B, D, H, W)
        out = out.view(B, D, H, W)
        return out

# --- Refinement Stage with Edge Guidance (from V2.5 Phase 2) ---
class RefinementStage(nn.Module):
    def __init__(self, guidance_channels, scale_factor, use_edge_guidance=False):
        super().__init__()
        self.scale_factor = scale_factor
        self.use_edge_guidance = use_edge_guidance
        extra = 1 if use_edge_guidance else 0
        self.net = nn.Sequential(
            nn.Conv2d(1 + guidance_channels + extra, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1)
        )
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def _edge_map(self, img):
        gx = F.conv2d(img, self.kx, padding=1)
        gy = F.conv2d(img, self.ky, padding=1)
        return torch.abs(gx) + torch.abs(gy)       # <-- NEU (L1-Trick)

    # In class RefinementStage(nn.Module):
    def forward(self, disparity_low, guidance, max_disp, gray_img=None): # <--- NEU: max_disp hinzugefügt
        disparity_up = F.interpolate(
            disparity_low, scale_factor=self.scale_factor,
            mode='bilinear', align_corners=False
        ) * self.scale_factor
        
        # ✅ PHYSIKALISCH FUNDIERTE NORMALISIERUNG
        # Wir bringen die Disparität für das CNN auf einen Prozentwert (0.0 bis 1.0)
        norm_disp = disparity_up / max_disp
        
        # Das CNN kriegt jetzt die normierte Disparität + die Guidance-Features
        inp = [norm_disp, guidance]
        
        if self.use_edge_guidance:
            assert gray_img is not None
            if gray_img.shape[-2:] != disparity_up.shape[-2:]:
                gray_img = F.interpolate(gray_img, size=disparity_up.shape[-2:],
                                         mode='bilinear', align_corners=False)
            inp.append(self._edge_map(gray_img))
            
        # Das berechnete Detail-Residual wird zur ORIGINALEN (unskalierten) Disparität addiert!
        return F.relu(disparity_up + self.net(torch.cat(inp, dim=1)))

# --- Stereo Head with Context Network ---
class HierarchicalStereoHead(nn.Module):
    def __init__(self, ch_s8, ch_s4, max_disp_s8, use_normals=True):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        self.use_normals = use_normals
        # ====================================================================
        # 🚨 PATCH V3.0: CoordConv für absolutes räumliches Bewusstsein!
        # kernel_size=1, padding=0 sorgt dafür, dass deine Architektur 
        # exakt gleich bleibt, nur dass X/Y elegant miteingemischt werden.
        # ====================================================================
        
        # 🚨 V3.1 GEOMETRY UPGRADE: 
        # Wir fügen geo_features (32 ch) hinzu. 
        # Für s8 müssen wir sie erst poolen, für s4 passen sie direkt.
        self.reduce_s8 = CoordConv2d(ch_s8 + 32, 32, kernel_size=1, padding=0, bias=False)


        # S4 Guidance: Normalen (3) + Geo (32) + Backbone (ch_s4)
        s4_guidance_ch = ch_s4 + 32 + (3 if use_normals else 0)
        self.reduce_s4 = CoordConv2d(s4_guidance_ch, 32, kernel_size=1, padding=0, bias=False)
      
        self.stereo_coarse = CoarseCostVolume(max_disp=self.max_disp_s8, in_channels=32)
        self.stereo_refine_s4 = RefinementStage(guidance_channels=32, scale_factor=2.0)
        self.stereo_refine_s1 = RefinementStage(guidance_channels=1, scale_factor=4.0, use_edge_guidance=True)
        
        self.register_buffer('disp_reg', torch.arange(self.max_disp_s8, dtype=torch.float32).view(1, -1, 1, 1))
        self.temperature = 0.7
        self.context_weight = 0.8
        
        # ====================================================================
        # 🚨 PATCH V3.0: Refactoring des Context-Blocks (Fix 2)
        # Sauberer Code, nutzt DWSepConv für die teure mittlere Schicht!
        # ====================================================================
        self.context = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),        # 1 auf 16: normale Conv
            nn.ReLU(inplace=True),
            DWSepConv(16, 16, 3, padding=1),       # 16 auf 16: schlankes DWSepConv!
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 3, padding=1)         # 16 auf 1: Output
        )

        # Lernbare Parameter für GeometryStem
        self.geo_downsample = nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, bias=False)

    # 🚨 V3.1: Signatur um geo_features_l und geo_features_r erweitert
    def forward(self, l_s8, r_s8, l_s4, l_img_raw, normals_s4=None, geo_features_l=None, geo_features_r=None):
        
        # --- SICHERHEITSNETZ (Für TensorBoard Tracer / Alte Checkpoints) ---
        if geo_features_l is None:
            B, _, H_s4, W_s4 = l_s4.shape
            geo_features_l = torch.zeros(B, 32, H_s4, W_s4, device=l_s4.device, dtype=l_s4.dtype)
        if geo_features_r is None:
            B, _, H_s4, W_s4 = l_s4.shape
            geo_features_r = torch.zeros(B, 32, H_s4, W_s4, device=l_s4.device, dtype=l_s4.dtype)

        # --- STUFE 1: S8 Coarse Matching ---
        # Da geo_features auf s4 (z.B. 160x120) sind, poolen wir sie für s8 (80x60)
        #geo_l_s8 = F.avg_pool2d(geo_features_l, kernel_size=2, stride=2)
        #geo_r_s8 = F.avg_pool2d(geo_features_r, kernel_size=2, stride=2)
        
        # geo_downsample statt avg_pool2d, um die Kantenschärfe zu erhalten
        geo_l_s8 = self.geo_downsample(geo_features_l)
        geo_r_s8 = self.geo_downsample(geo_features_r)
        
        # Jetzt mit Backbone-Features mischen (+ 32 Kanäle!)
        feat_l_s8 = self.reduce_s8(torch.cat([l_s8, geo_l_s8], dim=1))
        feat_r_s8 = self.reduce_s8(torch.cat([r_s8, geo_r_s8], dim=1))
        
        # --- STUFE 2: S4 Refinement Guidance ---
        if self.use_normals:
            if normals_s4 is not None:
                norm_in = normals_s4
            else:
                norm_in = torch.zeros(l_s4.size(0), 3, l_s4.size(2), l_s4.size(3), device=l_s4.device, dtype=l_s4.dtype)
            
            # Alle drei Quellen: Backbone (s4) + GeoStem (s4) + Normals (s4)
            l_s4_combined = torch.cat([l_s4, geo_features_l, norm_in], dim=1)
        else:
            l_s4_combined = torch.cat([l_s4, geo_features_l], dim=1)
            
        feat_l_s4 = self.reduce_s4(l_s4_combined)
        
        # --- STUFE 3: Stereo Prozess ---
        vol_s8 = self.stereo_coarse(feat_l_s8, feat_r_s8)
        
        # Kontext-Netzwerk
        B, D, H, W = vol_s8.shape
        vol_reshaped = vol_s8.view(B * D, 1, H, W)
        vol_ctx = self.context(vol_reshaped).view(B, D, H, W)
        
        vol_s8 = self.context_weight * vol_ctx + (1.0 - self.context_weight) * vol_s8
        
        prob_s8 = F.softmax(vol_s8 / self.temperature, dim=1)
        disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
        
        max_disp_s4 = self.max_disp_s8 * 2.0
        disp_s4 = self.stereo_refine_s4(disp_s8, feat_l_s4, max_disp=max_disp_s4)
        
        max_disp_s1 = max_disp_s4 * 4.0
        final_disp = self.stereo_refine_s1(disp_s4, l_img_raw, max_disp=max_disp_s1, gray_img=l_img_raw)
        
        return final_disp, disp_s8

# ✅ V3.0 Normals Head - Multi-Scale Fusion (s4 + s8), Up-Sampling, Image-Guided Refinement, CoordConv
class NormalsHead(nn.Module):
    def __init__(self, ch_s4, ch_s8):
        super().__init__()
        # Stage 1: Multi-Scale Fusion (s4 + s8)
        self.s8_adapt = nn.Conv2d(ch_s8, 32, kernel_size=1)
        
        # 🚨 V3.1 FIX: Backbone (ch_s4) + s8_adapt (32) + geo_features (32)
        fused_ch = ch_s4 + 32 + 32 
        
        # ====================================================================
        # 🚨 PATCH V3.0: CoordConv für den NormalsHead!
        # ====================================================================
        self.stage1 = nn.Sequential(
            CoordConv2d(fused_ch, 96, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.Conv2d(96, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(inplace=True)
        )
        self.coarse_out = nn.Conv2d(32, 3, 3, padding=1)

        # Stage 3: Image-Guided Refinement
        # Inputs: 3 (Coarse Normals) + 1 (Gray Img) + 1 (Sobel Edges) = 5
        self.refiner = nn.Sequential(
            nn.Conv2d(5, 32, 3, padding=1),
            
            # ====================================================================
            # 🚨 PATCH V3.0: Clean Code mit DWSepConv! 
            # (Ersetzt die 4 manuellen Layer von vorher)
            # ====================================================================
            DWSepConv(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            
            DWSepConv(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(32, 3, 3, padding=1)
        )
        
        # Fest verdrahtete Sobel-Filter (keine trainierbaren Parameter)
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def get_edges(self, img):
        gx = F.conv2d(img, self.sobel_x, padding=1)
        gy = F.conv2d(img, self.sobel_y, padding=1)
        # 🚨 100% Hailo-Safe: L1-Norm (Manhattan-Distanz) statt Wurzel
        # Das löst auch das FP16 NaN-Problem automatisch, da es keine Wurzel mehr gibt!
        return torch.abs(gx) + torch.abs(gy)

    def forward(self, l_s4, l_s8, gray_img, geo_features=None):
        # 1. s8 Features anpassen und auf s4 Größe hochziehen
        s8_adapted = self.s8_adapt(l_s8)
        s8_adapted = F.interpolate(s8_adapted, size=l_s4.shape[2:], mode='bilinear', align_corners=False)
        
        # 2. 🚨 V3.1 Alle drei Quellen zusammenbauen!
        if geo_features is not None:
            l_s4_combined = torch.cat([l_s4, s8_adapted, geo_features], dim=1)
        else:
            # 🚨 FIX: Hartkodierte 32 Kanäle für den Dummy-Tensor, falls TensorBoard testet
            dummy_geo = torch.zeros(l_s4.size(0), 32, l_s4.size(2), l_s4.size(3), 
                                    device=l_s4.device, dtype=l_s4.dtype)
            l_s4_combined = torch.cat([l_s4, s8_adapted, dummy_geo], dim=1)
        
        # 3. Ab in den CoordConv
        feat_s4 = self.stage1(l_s4_combined)
        
        # Normale berechnen (Coarse)
        coarse_normals_s4 = self.coarse_out(feat_s4)
        coarse_normals_s4 = F.normalize(coarse_normals_s4, dim=1)
        
        # Upsample für Stage 3
        normals_coarse = F.interpolate(coarse_normals_s4, size=gray_img.shape[2:], mode='bilinear', align_corners=False)
        
        # Stage 3: Image-Guided Refinement
        edges = self.get_edges(gray_img)
        refine_in = torch.cat([normals_coarse, gray_img, edges], dim=1)
        refined = self.refiner(refine_in)
        
        # Residual-Verbindung & Normalisierung
        normals_s1 = normals_coarse + refined
        # 🚨 FP16-Safe Epsilon explizit setzen (Standard ist 1e-12 -> in FP16 ist das 0.0 -> NaN!)
        normals_s1 = F.normalize(normals_s1, p=2, dim=1, eps=1e-4)
        normals_s4 = F.normalize(coarse_normals_s4, p=2, dim=1, eps=1e-4)
        
        return normals_s1, normals_s4

# ✅ V3.1: LRASPPHead with Normals input + 6 classes
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes, normals_ch=3):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.scale_high = nn.Sequential(
            nn.AvgPool2d(kernel_size=(30, 40)),
            nn.Conv2d(high_ch, 128, 1, bias=False),
            nn.Sigmoid()
        )
        # ✅ V2.9: low_classifier takes backbone features + predicted normals
        self.low_classifier = nn.Conv2d(low_ch + normals_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)
        # Dilated conv also gets normals (activated in Phase 2)
        self.mid_classifier = nn.Conv2d(128 + normals_ch, num_classes, 3, padding=2, dilation=2)
        self.use_mid = True
        
    def forward(self, x_low, x_high, normals_s4=None):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        out = F.interpolate(out, scale_factor=4.0, mode='bilinear', align_corners=False)
        
        if normals_s4 is not None:
            low_in = torch.cat([x_low, normals_s4], dim=1)
        else:
            # Make it Hailo compatible (kein F.pad auf Channels):
            dummy_normals = torch.zeros(x_low.size(0), 3, x_low.size(2), x_low.size(3), device=x_low.device)
            low_in = torch.cat([x_low, dummy_normals], dim=1)
            
        result = self.low_classifier(low_in) + self.high_classifier(out)
        
        if self.use_mid:
            if normals_s4 is not None:
                mid_in = torch.cat([out, normals_s4], dim=1)
            else:
                # ✅ KORRIGIERT: Auch hier dummy_normals mit torch.cat statt F.pad
                dummy_normals_mid = torch.zeros(out.size(0), 3, out.size(2), out.size(3), device=out.device)
                mid_in = torch.cat([out, dummy_normals_mid], dim=1)
                
            result = result + self.mid_classifier(mid_in)   
        return result
        
# --- YOLO Heads (unchanged structure, 40 classes) ---
class DecoupledHead(nn.Module):
    def __init__(self, ch_in, num_classes, width=128):
        super().__init__()
        
        self.coord_conv_cls = CoordConv2d(ch_in, width, kernel_size=3, padding=1)
        self.coord_conv_reg = CoordConv2d(ch_in, width, kernel_size=3, padding=1)
        
        # 🚨 PATCH V3.0: RepConv integriert die ReLU bereits intern!
        self.cls_convs = nn.Sequential(
            self.coord_conv_cls,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.reg_convs = nn.Sequential(
            self.coord_conv_reg,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.cls_pred = nn.Conv2d(width, num_classes, 1)
        self.reg_pred = nn.Conv2d(width, 4, 1)
        self.obj_pred = nn.Conv2d(width, 1, 1)
    def forward(self, x):
        cls_feat = self.cls_convs(x); reg_feat = self.reg_convs(x)
        return torch.cat([self.reg_pred(reg_feat), self.obj_pred(reg_feat), self.cls_pred(cls_feat)], dim=1)

class YOLOHead(nn.Module):
    def __init__(self, fpn_ch=FPN_CH, num_classes=40):
        super().__init__()
        self.head_s8  = DecoupledHead(fpn_ch, num_classes, width=128)
        self.head_s16 = DecoupledHead(fpn_ch, num_classes, width=128)
        self.head_s32 = DecoupledHead(fpn_ch, num_classes, width=128)
    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]

# =====================================================================
# ✅ V3.1: FusedHexapodModel — 1-Channel Input, 5 Outputs
# =====================================================================
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True,
                                           features_only=True, out_indices=(1, 2, 3, 4))
        feat_info = self.backbone.feature_info.channels()
        ch_s4, ch_s8, ch_s16, ch_s32 = feat_info

        # ✅ V2.9: Patch first conv to 1-channel input
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride, padding=old_conv.padding, bias=False)
        # Merge RGB weights via luminance formula
        with torch.no_grad():
            w = old_conv.weight.data  # [C_out, 3, kH, kW]
            new_conv.weight.data = w[:, 0:1]*0.299 + w[:, 1:2]*0.587 + w[:, 2:3]*0.114
        self.backbone.conv_stem = new_conv
        print(f"  ✅ Backbone first conv: 3→1 channel (luminance merge)")
        
        # 🚨 PATCH V3.0: SPPF initialisieren (nimmt ch_s32 dynamisch von timm!)
        self.sppf = SPPF(c1=ch_s32, c2=ch_s32, k=5)
        print(f"  ✅ SPPF Module injected at s32 ({ch_s32} channels)")
        
        disp_steps = config.get('internal_disp_steps', 48)
        self.stereo_head = HierarchicalStereoHead(ch_s8, ch_s4, max_disp_s8=disp_steps)
        self.normals_head = NormalsHead(ch_s4, ch_s8)
        self.seg_head = LRASPPHead(ch_s4, ch_s16, config['num_seg_classes'], normals_ch=3)
        self.fpn_neck = LightFPNNeck(ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH)

        # 🚨 PATCH V3.0: CBAM Attention für die FPN-Outputs
        self.cbam_s8  = CBAM(FPN_CH)
        self.cbam_s16 = CBAM(FPN_CH)
        self.cbam_s32 = CBAM(FPN_CH)
        print(f"  ✅ CBAM Attention Modules injected after FPN")

        self.geo_stem = GeometryStem(in_ch=24, out_ch=32)
        print(f"  ✅ Geometry Stem Modules injected after Backbone")

        self.yolo_head = YOLOHead(fpn_ch=FPN_CH, num_classes=config['num_det_classes'])

        total_params = sum(p.numel() for p in self.parameters())
        print(f"  Total parameters: {total_params:,}")
        print(f"  Normals Head: {sum(p.numel() for p in self.normals_head.parameters()):,}")
        print(f"  Seg Head: {sum(p.numel() for p in self.seg_head.parameters()):,}")
        print(f"  YOLO Head: {sum(p.numel() for p in self.yolo_head.parameters()):,}")

    def forward(self, x_left, x_right, use_normals_for_stereo=False):
        # Backbone & FPN
        features_l = self.backbone(x_left)
        # ====================================================================
        # 🚨 NEU V3.1: GEOMETRY STEM (High-Res Pfad)
        # Nutzt features_l[0] (s4 / 160x120), um scharfe Kanten-Features zu extrahieren
        # ====================================================================
        geo_feat_l = self.geo_stem(features_l[0])
        # Beide Auflösungen vom Normals-Head abgreifen (Neu: mit s8 und gray_img)
        # Normals bekommt jetzt die Geo-Features zusätzlich
        normals_s1, normals_s4 = self.normals_head(
            features_l[0], 
            features_l[1], 
            x_left, 
            geo_features=geo_feat_l # 👈 NEU: High-Res Support
        )
        
        final_disp, disp_s8 = None, None
        
        # ✅ FIX: Nur Stereo ausführen, wenn wir auch ein rechtes Bild haben (TartanAir)
        if x_right is not None:
            with torch.no_grad():
                features_r = self.backbone(x_right)
                # Auch für das rechte Bild brauchen wir die Geo-Features für das Matching!
                geo_feat_r = self.geo_stem(features_r[0])
            
            # 🚨 FIX: normals_s4.detach() verhindert, dass Stereo-Gradienten den NormalsHead zerstören!
            normals_s4_for_stereo = normals_s4.detach() if (use_normals_for_stereo and normals_s4 is not None) else None
            
            # Stereo Head (nutzt jetzt die entkoppelten Normalen UND das Graustufenbild)
            # Stereo Head bekommt jetzt geo_feat_l UND geo_feat_r
            final_disp, disp_s8 = self.stereo_head(
                features_l[1],    # l_s8
                features_r[1],    # r_s8
                features_l[0],    # l_s4
                x_left,           # l_img_raw
                normals_s4=normals_s4_for_stereo,
                geo_features_l=geo_feat_l, # 👈 NEU: Linke Geo-Features
                geo_features_r=geo_feat_r  # 👈 NEU: Rechte Geo-Features
            )
        else:
            final_disp, disp_s8 = None, None
            
        # Seg bekommt ebenfalls die S4-Normalen (160x120)
        normals_for_others = normals_s4.detach() if (use_normals_for_stereo and normals_s4 is not None) else None
        seg = self.seg_head(features_l[0], features_l[2], normals_s4=normals_for_others)

        # 🚨 PATCH V3.0: SPPF auf die tiefste Ebene anwenden
        f_s32_sppf = self.sppf(features_l[3])
        # FPN_Neck mit der gepatchten s32-Map aufrufen
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(features_l[1], features_l[2], f_s32_sppf)
        
        # ====================================================================
        # 🚨 PATCH V3.0: CBAM Attention vor dem YOLO-Head anwenden!
        # ====================================================================
        fpn_s8_att  = self.cbam_s8(fpn_s8)
        fpn_s16_att = self.cbam_s16(fpn_s16)
        fpn_s32_att = self.cbam_s32(fpn_s32)

        # YOLO bekommt jetzt die gefilterten Attention-Features!
        det = self.yolo_head(fpn_s8_att, fpn_s16_att, fpn_s32_att)

        return final_disp, seg, det, disp_s8, normals_s1

model = FusedHexapodModel(CONFIG).to(DEVICE)
print(f"\n✅ V3.10 Model created on {DEVICE}")


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  ✅ Backbone first conv: 3→1 channel (luminance merge)
  ✅ SPPF Module injected at s32 (960 channels)
  ✅ CBAM Attention Modules injected after FPN
  ✅ Geometry Stem Modules injected after Backbone
  Total parameters: 7,237,261
  Normals Head: 158,790
  Seg Head: 36,950
  YOLO Head: 1,462,791

✅ V3.10 Model created on cuda


In [3]:
import torch
import os
import copy

# =====================================================================
# 🛠️ KONFIGURATION
# =====================================================================
checkpoint_paths = [
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth"
]

# Das fertige, gefaltete Modell für den Hailo-Export
output_path_folded = "checkpoints/checkpoint_v3_1_swa_folded.pth"

# =====================================================================
# 🚀 SWA + FOLDING LOGIK
# =====================================================================
def create_swa_and_fold(filepaths, out_path):
    print(f"🔄 1. Starte SWA: Mittle {len(filepaths)} Checkpoints...")
    
    swa_state_dict = None
    num_checkpoints = len(filepaths)
    
    for path in filepaths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"❌ Checkpoint nicht gefunden: {path}")
            
        print(f"📥 Lade: {path}")
        ckpt = torch.load(path, map_location='cpu')
        model_state = ckpt.get('model_state_dict', ckpt)
        
        if swa_state_dict is None:
            swa_state_dict = {k: v.clone() for k, v in model_state.items()}
        else:
            for k in swa_state_dict.keys():
                if k in model_state:
                    swa_state_dict[k] += model_state[k]
    
    # Durchschnitt berechnen
    for k in swa_state_dict.keys():
        if swa_state_dict[k].is_floating_point():
            swa_state_dict[k].div_(num_checkpoints)
        else:
            swa_state_dict[k] = torch.div(swa_state_dict[k], num_checkpoints, rounding_mode='floor')
            
    print("✅ Mittelung abgeschlossen.")

    # -----------------------------------------------------------------
    # 🏗️ MODELL LADEN UND FALTEN (switch_to_deploy)
    # -----------------------------------------------------------------
    print("🏗️ 2. Lade SWA-Gewichte in Modellstruktur für Reparametrisierung...")
    
    # Wir brauchen eine Instanz deines Modells im Training-Mode (Deploy=False)
    # Annahme: Deine Modellklasse heißt 'FusedHexapodModel'
    # WICHTIG: Ersetze dies durch deine tatsächliche Initialisierung!
    # model = FusedHexapodModel(deploy=False) 
    
    # Falls das Modell schon im Notebook existiert, nutzen wir es direkt:
    try:
        model.load_state_dict(swa_state_dict, strict=True)
    except NameError:
        print("❌ Fehler: 'model' Instanz nicht gefunden. Bitte führe die Modell-Definition vorher aus.")
        return

    model.eval() # Wichtig für switch_to_deploy
    
    print("🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...")
    folded_count = 0
    for m in model.modules():
        if hasattr(m, 'switch_to_deploy'):
            m.switch_to_deploy()
            folded_count += 1
    
    print(f"✨ {folded_count} Layer erfolgreich gefaltet!")

    # -----------------------------------------------------------------
    # 💾 SPEICHERN FÜR ONNX
    # -----------------------------------------------------------------
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    
    # Wir speichern NUR das state_dict des nun gefalteten Modells
    torch.save(model.state_dict(), out_path)
    
    print(f"💾 Finales Modell gespeichert: {out_path}")
    

# Ausführen
if __name__ == "__main__":
    # Stelle sicher, dass 'model' in deinem Notebook definiert ist, bevor du das hier rufst!
    create_swa_and_fold(checkpoint_paths, output_path_folded)

🔄 1. Starte SWA: Mittle 6 Checkpoints...
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth


/tmp/ipykernel_2622373/1344699933.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location='cpu')


📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth
✅ Mittelung abgeschlossen.
🏗️ 2. Lade SWA-Gewichte in Modellstruktur für Reparametrisierung...
🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...
✨ 12 Layer erfolgreich gefaltet!
💾 Finales Modell gespeichert: checkpoints/checkpoint_v3_1_swa_folded.pth


In [4]:
# =====================================================================
# 🛠️ HAILO-OPTIMIERTER ONNX EXPORT
# =====================================================================
import torch.onnx

onnx_path = "checkpoints/hexapod_v3_1_swa_final.onnx"
model.eval()

# Schritt 1: Das Argument 'use_normals_for_stereo' im Modell hart fixieren
# Wir patchen die Methode kurzzeitig, damit der ONNX-Tracer keine Booleans sieht
original_forward = model.forward
def forward_fixed(x_left, x_right):
    # Wir rufen das Original auf, aber erzwingen den Schalter auf True
    return original_forward(x_left, x_right, use_normals_for_stereo=True)

model.forward = forward_fixed

# Schritt 2: Dummy-Inputs (WICHTIG: Auf CPU bleiben für saubereren Trace)
dummy_l = torch.randn(1, 1, 480, 640)
dummy_r = torch.randn(1, 1, 480, 640)
model.cpu() 

print(f"📦 Exportiere HAILO-Version nach {onnx_path}...")

with torch.no_grad():
    torch.onnx.export(
        model,
        (dummy_l, dummy_r),
        onnx_path,
        export_params=True,
        opset_version=11, # 👈 VERSUCH OPSET 11 (Oft stabiler für Hailo DFC 3.x/5.x)
        do_constant_folding=True,
        input_names=['input_left', 'input_right'],
        output_names=['disparity', 'segmentation', 'detections', 'disp_s8', 'normals'],
        # 🚨 Das hier ist oft der Retter:
        operator_export_type=torch.onnx.OperatorExportTypes.ONNX_ATEN_FALLBACK 
    )

# Modell wieder zurück auf Original setzen
model.forward = original_forward
model.to(DEVICE)

print("✅ ONNX-Export erfolgreich!")
print("💡 Tipp: Öffne die Datei jetzt in 'netron.app', um die Struktur zu prüfen.")

📦 Exportiere HAILO-Version nach checkpoints/hexapod_v3_1_swa_final.onnx...


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:485: FutureWarning: Setting `operator_export_type` to something other than default is deprecated. The option will be removed in a future release.
  warnings.warn(
/tmp/ipykernel_2622166/4087738642.py:282: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if gray_img.shape[-2:] != disparity_up.shape[-2:]:


✅ ONNX-Export erfolgreich!
💡 Tipp: Öffne die Datei jetzt in 'netron.app', um die Struktur zu prüfen.


In [5]:
import torch
import torch.onnx
import copy

onnx_path = "checkpoints/hexapod_v3_1_swa_final.onnx"

# 1. Frische Instanz (mit den neuen CoordConv Buffern!)
clean_model = FusedHexapodModel(CONFIG)

# 2. Falten (Reparametrisieren)
for m in clean_model.modules():
    if hasattr(m, 'switch_to_deploy'):
        m.switch_to_deploy()

# 3. Gewichte aus deinem SWA-Folded File laden
# (Hier laden wir die Gewichte in die neue Struktur)
missing, unexpected = clean_model.load_state_dict(torch.load("checkpoints/checkpoint_v3_1_swa_folded.pth", map_location='cpu'))
print(f"Missing (nur Buffers, OK): {missing}")
clean_model.eval().cpu()

def export_forward(l, r):
    # Wir rufen das original forward auf
    # Da wir uns innerhalb der Zelle befinden, kennt die Funktion 'clean_model'
    # Wir erzwingen use_normals_for_stereo=True
    disp, seg, det_list, disp_s8, normals = clean_model.original_forward(l, r, use_normals_for_stereo=True)
    
    # YOLO-Listen-Fix: Wir entpacken die 3 Scales
    yolo_s8, yolo_s16, yolo_s32 = det_list
    
    # Wir geben jetzt 6 einzelne Tensoren zurück (keine Listen, keine Dicts!)
    return disp, seg, normals, disp_s8, yolo_s8, yolo_s16, yolo_s32

# Wir sichern das original forward und setzen die Umleitung
clean_model.original_forward = clean_model.forward 
clean_model.forward = export_forward

onnx_path = "checkpoints/hexapod_v3_1_hardware.onnx"
dummy_l = torch.randn(1, 1, 480, 640)
dummy_r = torch.randn(1, 1, 480, 640)

# Die Namen müssen exakt mit der Anzahl der Rückgabewerte in export_forward übereinstimmen
output_names = ['disparity', 'segmentation', 'normals', 'disp_s8', 'yolo_s8', 'yolo_s16', 'yolo_s32']

print(f"📦 Exportiere Hardware-Modell nach {onnx_path}...")

torch.onnx.export(
    clean_model,
    (dummy_l, dummy_r),
    onnx_path,
    export_params=True,
    opset_version=13, # Wichtig für die CoordConv-Buffer
    do_constant_folding=True,
    input_names=['input_left', 'input_right'],
    output_names=output_names,
    keep_initializers_as_inputs=False
)

print("✅ Fertig! Jetzt ab zum Hailo-Parser.")

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
/tmp/ipykernel_2622373/345351692.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file

  ✅ Backbone first conv: 3→1 channel (luminance merge)
  ✅ SPPF Module injected at s32 (960 channels)
  ✅ CBAM Attention Modules injected after FPN
  ✅ Geometry Stem Modules injected after Backbone
  Total parameters: 7,237,261
  Normals Head: 158,790
  Seg Head: 36,950
  YOLO Head: 1,462,791
Missing (nur Buffers, OK): []
📦 Exportiere Hardware-Modell nach checkpoints/hexapod_v3_1_hardware.onnx...


/tmp/ipykernel_2622373/2106743034.py:298: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if gray_img.shape[-2:] != disparity_up.shape[-2:]:


✅ Fertig! Jetzt ab zum Hailo-Parser.


In [8]:
import onnx
from onnx import numpy_helper

model = onnx.load('checkpoints/hexapod_v3_1_swa_final.onnx')
onnx.checker.check_model(model)
print(f"Nodes: {len(model.graph.node)}, Initializers: {len(model.graph.initializer)}")

# Alle Initializer-Namen sammeln
init_names = {i.name for i in model.graph.initializer}
input_names = {i.name for i in model.graph.input}
node_outputs = {o for n in model.graph.node for o in n.output}

# Finde Conv-Nodes deren Gewichts-Input kein Initializer ist
print("\n=== Conv nodes with non-initializer weight inputs ===")
for i, node in enumerate(model.graph.node):
    if node.op_type == 'Conv':
        # Conv inputs: [data, weight, bias(optional)]
        if len(node.input) >= 2:
            weight_input = node.input[1]
            if weight_input not in init_names:
                # Find the producer node
                producer = None
                for n in model.graph.node:
                    if weight_input in n.output:
                        producer = n
                        break
                prod_info = f"{producer.op_type} (attrs={len(producer.attribute)})" if producer else "NONE"
                print(f"  Node {i}: {node.name}")
                print(f"    weight input: '{weight_input}' → produced by: {prod_info}")
                if producer:
                    print(f"    producer inputs: {list(producer.input)}")

# Auch Constant-Nodes ohne shape prüfen
print("\n=== Suspicious Constant nodes (no dims) ===")
for node in model.graph.node:
    if node.op_type == 'Constant':
        for attr in node.attribute:
            if attr.name == 'value' and hasattr(attr, 't'):
                if len(attr.t.dims) == 0 and attr.t.data_type != 0:
                    print(f"  {node.name} → output: {node.output[0]}, scalar constant")

Nodes: 1891, Initializers: 249

=== Conv nodes with non-initializer weight inputs ===
  Node 310: /normals_head/Conv
    weight input: 'normals_head.sobel_x' → produced by: Identity (attrs=0)
    producer inputs: ['stereo_head.stereo_refine_s1.kx']
  Node 311: /normals_head/Conv_1
    weight input: 'normals_head.sobel_y' → produced by: Identity (attrs=0)
    producer inputs: ['stereo_head.stereo_refine_s1.ky']
  Node 337: /backbone/conv_stem_1/Conv
    weight input: 'onnx::Conv_3085' → produced by: Identity (attrs=0)
    producer inputs: ['onnx::Conv_2947']
  Node 340: /backbone/blocks.0/blocks.0.0/conv_dw_1/Conv
    weight input: 'onnx::Conv_3088' → produced by: Identity (attrs=0)
    producer inputs: ['onnx::Conv_2950']
  Node 342: /backbone/blocks.0/blocks.0.0/conv_pw_1/Conv
    weight input: 'onnx::Conv_3091' → produced by: Identity (attrs=0)
    producer inputs: ['onnx::Conv_2953']
  Node 344: /backbone/blocks.1/blocks.1.0/conv_pw_1/Conv
    weight input: 'onnx::Conv_3094' → produ

In [10]:
import onnx
model = onnx.load('checkpoints/hexapod_v3_1_simplified.onnx')

# Finde alle Outputs der reduce_s8 und reduce_s4 Convs
for node in model.graph.node:
    name = node.name
    if any(k in name for k in ['reduce_s8', 'reduce_s4', 'seg_head/Add',
                                 'normals_head/Div', 'normals_head/Add',
                                 'fpn_neck', 'cbam', 'yolo_head']):
        if node.op_type in ['Conv', 'Add', 'Div', 'Relu', 'Sigmoid']:
            print(f"{node.op_type:10s} {name}")
            print(f"           outputs: {list(node.output)}")

Div        /normals_head/Div
           outputs: ['/normals_head/Div_output_0']
Add        /normals_head/Add
           outputs: ['/normals_head/Add_output_0']
Add        /normals_head/Add_1
           outputs: ['/normals_head/Add_1_output_0']
Div        /normals_head/Div_1
           outputs: ['837']
Div        /normals_head/Div_2
           outputs: ['/normals_head/Div_2_output_0']
Conv       /stereo_head/reduce_s8/conv/Conv
           outputs: ['/stereo_head/reduce_s8/conv/Conv_output_0']
Conv       /stereo_head/reduce_s8/conv_1/Conv
           outputs: ['/stereo_head/reduce_s8/conv_1/Conv_output_0']
Conv       /stereo_head/reduce_s4/conv/Conv
           outputs: ['/stereo_head/reduce_s4/conv/Conv_output_0']
Add        /seg_head/Add
           outputs: ['/seg_head/Add_output_0']
Add        /seg_head/Add_1
           outputs: ['segmentation']
Conv       /fpn_neck/lat_s32/Conv
           outputs: ['/fpn_neck/lat_s32/Conv_output_0']
Conv       /fpn_neck/lat_s16/Conv
           outputs: